In [92]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
#from natsort import natsort as ns
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report, roc_auc_score

In [93]:
%store -r dado_frame 

In [94]:
#separar features e o targete

features_coluna = [coluna for coluna in dado_frame.columns if coluna not in ['timestamp','target_iALL_PS']]   

In [121]:
X = dado_frame[features_coluna]
y = (dado_frame['target_iALL_PS'] == ('ANORMAL')).astype(int)

In [118]:
print(X)

        TAG_iALL_PS_00  TAG_iALL_PS_01  TAG_iALL_PS_02  TAG_iALL_PS_03  \
0             4.548754       90.886874       58.698896       89.301134   
1             7.887998       56.555373       80.802810      120.898222   
2             4.975919      182.086958       87.273632        9.914782   
3             6.304142       58.417235       75.059520       64.167463   
4             1.671733      108.946809       94.910470       14.551922   
...                ...             ...             ...             ...   
220315        6.066461      128.157145      144.452803       66.621842   
220316        4.057784      -31.186502       49.775117       69.397205   
220317        6.751912       80.433446      107.038506      137.923928   
220318        2.084228      192.076427      125.030042       47.801000   
220319        4.558682       16.293402       45.667827       95.239698   

        TAG_iALL_PS_04  TAG_iALL_PS_05  TAG_iALL_PS_06  TAG_iALL_PS_07  \
0          1011.733181       97.28488

In [120]:
print(y)

0         NORMAL
1         NORMAL
2         NORMAL
3         NORMAL
4         NORMAL
           ...  
220315    NORMAL
220316    NORMAL
220317    NORMAL
220318    NORMAL
220319    NORMAL
Name: target_iALL_PS, Length: 220320, dtype: object


In [98]:
print(X.shape)

(220320, 51)


In [99]:
print(y.shape)

(220320,)


In [100]:
soma_y = y.sum() 
media_y = y.mean() * 100
print(soma_y)
print(media_y)

14484
6.5740740740740735


In [101]:
#dividir os dados em treino e teste usando amotragem estratificada ela garante que a 
#a proporçção de classes seja mantida, tando no conjunto de treino quanto no conjunto de testes.
# usa uma quantiidade de amostrar para ter o menor  erro. 

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for (treino_index, teste_index) in sss.split(X,y):
    X_treino, X_teste = X.iloc[treino_index], X.iloc[teste_index]
    y_treino, y_teste = y.iloc[treino_index], y.iloc[teste_index] 

 
y_treino_soma = y_treino.sum()
y_teste_soma = y_teste.sum() 

print(y_treino_soma)
print(y_teste_soma)


11587
2897


In [102]:
#aplicar o modelo de arvore de decisão

arvore_decisao = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42) 

arvore_decisao.fit(X_treino, y_treino)

y_predicao_arvore = arvore_decisao.predict(X_teste)
y_probilodade_arvore = arvore_decisao.predict_proba(X_teste)[:,1]

print("Arvore de decisaõ:")
print(classification_report(y_teste, y_predicao_arvore,target_names=['NORMAL', 'ANORMAL']))

Arvore de decisaõ:
              precision    recall  f1-score   support

      NORMAL       1.00      0.99      0.99     41167
     ANORMAL       0.82      0.98      0.90      2897

    accuracy                           0.98     44064
   macro avg       0.91      0.98      0.94     44064
weighted avg       0.99      0.98      0.99     44064



In [103]:
# Floresta aleatória
floresta_aleatoria = RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=10, class_weight='balanced', random_state=42, n_jobs=-1 )

floresta_aleatoria.fit(X_treino, y_treino)

y_predicao_floresta = floresta_aleatoria.predict(X_teste)
y_probilodade_floresta = floresta_aleatoria.predict_proba(X_teste)[:,1]

print("Floresta aleatória:")
print(classification_report(y_teste, y_predicao_floresta, target_names=['NORMAL', 'ANORMAL']))
 


Floresta aleatória:
              precision    recall  f1-score   support

      NORMAL       1.00      0.99      1.00     41167
     ANORMAL       0.90      0.99      0.94      2897

    accuracy                           0.99     44064
   macro avg       0.95      0.99      0.97     44064
weighted avg       0.99      0.99      0.99     44064



In [104]:
roc_arvore = roc_auc_score(y_teste, y_probilodade_floresta)
roc_floresta = roc_auc_score(y_teste, y_probilodade_floresta)

In [105]:

print(f"{roc_arvore:.4f}")
print(f"{roc_floresta:.4f}")

0.9983
0.9983
